# Controller Rollout Inference Analysis

Analyze sampled-controller rollout runs produced by `run_scripts/run_m3cot_rollout_inference.sh` / `lvar_scripts/infer_lvar_m3cot_rollouts.py`.

The notebook compares raw rollout accuracy, best-of-N majority-vote accuracy, oracle accuracy, random-rollout accuracy, answer diversity, decoded entropy, answer-option entropy, controller entropy, hidden-step entropy, and metric-selected rollout accuracy. Prefix rollout tracking is intentionally absent for these runs.


In [ ]:
%matplotlib inline

from collections import Counter
import json
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd
from IPython.display import display

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False

ROOT = Path.cwd()
if ROOT.name == "analysis":
    ROOT = ROOT.parent

# Point this at one run directory, an output root, or leave it recursive.
RUN_ROOT = ROOT / "outputs" / "inference"

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 140)

## Load Results

Each rollout run writes a summary JSON, raw rollout JSONL, accuracy-variant JSONL, and entropy sidecar JSON. The loader below discovers summary files and uses the paths recorded in each summary when available.

In [ ]:
def load_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def load_jsonl(path):
    rows = []
    path = Path(path)
    if not path.exists():
        return rows
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            stripped = line.strip()
            if not stripped:
                continue
            try:
                rows.append(json.loads(stripped))
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON in {path} line {line_number}: {exc}") from exc
    return rows


def discover_summaries(root):
    root = Path(root)
    if root.is_file() and root.name.endswith("_summary.json"):
        return [root]
    if not root.exists():
        return []
    return sorted(
        path for path in root.rglob("*_summary.json")
        if "rollout" in path.name or "controller_rollouts" in str(path)
    )


summary_paths = discover_summaries(RUN_ROOT)
print(f"Discovered {len(summary_paths)} rollout summary files under {RUN_ROOT}")
for path in summary_paths[:10]:
    print(" -", path.relative_to(ROOT) if path.is_relative_to(ROOT) else path)

if not summary_paths:
    print("No rollout summaries found yet. Run run_scripts/run_m3cot_rollout_inference.sh or set RUN_ROOT above.")

In [ ]:
summaries = []
rollout_rows = []
variant_rows = []
entropy_rows = []

for summary_path in summary_paths:
    summary = load_json(summary_path)
    run_id = str(summary_path.parent.relative_to(ROOT)) if summary_path.parent.is_relative_to(ROOT) else str(summary_path.parent)
    metrics = summary.get("metrics", {})
    summaries.append({
        "run_id": run_id,
        "summary_path": str(summary_path),
        "dataset_partition": summary.get("dataset_partition"),
        "num_examples": summary.get("num_examples"),
        "num_rollouts": summary.get("num_rollouts"),
        "controller_temperature": summary.get("controller_temperature"),
        "seed": summary.get("seed"),
        "coarse_context": summary.get("coarse_context"),
        **metrics,
    })

    rollout_path = Path(summary.get("rollout_predictions_path", ""))
    variant_path = Path(summary.get("accuracy_variants_path", ""))
    entropy_path = Path(summary.get("entropy_tracking_path", ""))
    if not rollout_path.is_absolute():
        rollout_path = ROOT / rollout_path
    if not variant_path.is_absolute():
        variant_path = ROOT / variant_path
    if not entropy_path.is_absolute():
        entropy_path = ROOT / entropy_path

    for row in load_jsonl(rollout_path):
        row["run_id"] = run_id
        rollout_rows.append(row)
    for row in load_jsonl(variant_path):
        row["run_id"] = run_id
        variant_rows.append(row)
    entropy_data = load_json(entropy_path) if entropy_path.exists() else []
    for row in entropy_data:
        row["run_id"] = run_id
        entropy_rows.append(row)

summary_df = pd.DataFrame(summaries)
rollouts_df = pd.DataFrame(rollout_rows)
variants_df = pd.DataFrame(variant_rows)
entropy_df = pd.DataFrame(entropy_rows)

print(f"Summaries: {len(summary_df):,}")
print(f"Rollout rows: {len(rollouts_df):,}")
print(f"Variant rows: {len(variants_df):,}")
print(f"Entropy rows: {len(entropy_df):,}")

## Run Summary

This table is the fastest check that the three aggregate accuracy variants were computed as expected.

In [ ]:
accuracy_cols = ["rollout_accuracy", "best_of_n_accuracy", "oracle_accuracy", "random_accuracy"]
if summary_df.empty:
    display(summary_df)
else:
    display_cols = [
        "run_id", "dataset_partition", "num_examples", "num_rollouts",
        "controller_temperature", "seed", "coarse_context", *accuracy_cols,
    ]
    display(summary_df[display_cols].sort_values(["dataset_partition", "controller_temperature", "num_rollouts"]).style.format({
        "rollout_accuracy": "{:.2%}",
        "best_of_n_accuracy": "{:.2%}",
        "oracle_accuracy": "{:.2%}",
        "random_accuracy": "{:.2%}",
    }))

    plot_df = summary_df.melt(
        id_vars=["run_id", "num_rollouts", "controller_temperature"],
        value_vars=accuracy_cols,
        var_name="metric",
        value_name="accuracy",
    )
    plt.figure(figsize=(10, max(4, 0.45 * len(summary_df))))
    if HAS_SEABORN:
        sns.barplot(data=plot_df, y="run_id", x="accuracy", hue="metric")
    else:
        for metric, group in plot_df.groupby("metric"):
            plt.scatter(group["accuracy"], group["run_id"], label=metric)
        plt.legend()
    plt.gca().xaxis.set_major_formatter(PercentFormatter(1.0))
    plt.title("Accuracy Variants By Run")
    plt.xlabel("Accuracy")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()

## Rollout-Level Behavior

Raw rollout accuracy treats every sampled controller trajectory as one prediction. The diversity columns show how often the controller sampling actually changes final answers.

In [ ]:
if rollouts_df.empty:
    display(rollouts_df)
else:
    rollouts_df["correct"] = rollouts_df["correct"].astype(bool)
    per_prompt = rollouts_df.groupby(["run_id", "example_id"]).agg(
        rollout_count=("rollout_idx", "count"),
        any_correct=("correct", "any"),
        random_accuracy_estimate=("correct", "mean"),
        unique_answers=("answer_key", "nunique"),
        majority_fraction=("answer_key", lambda values: Counter(values).most_common(1)[0][1] / len(values)),
        mean_decoded_entropy=("token_entropy_mean", "mean"),
        mean_hidden_entropy=("hidden_step_entropy_mean", "mean"),
        mean_controller_entropy=("controller_entropy_mean", "mean"),
    ).reset_index()
    run_prompt_summary = per_prompt.groupby("run_id").agg(
        prompts=("example_id", "count"),
        oracle_rate=("any_correct", "mean"),
        mean_unique_answers=("unique_answers", "mean"),
        answer_changed_rate=("unique_answers", lambda values: (values > 1).mean()),
        mean_majority_fraction=("majority_fraction", "mean"),
    ).reset_index()
    display(run_prompt_summary.style.format({
        "oracle_rate": "{:.2%}",
        "answer_changed_rate": "{:.2%}",
        "mean_unique_answers": "{:.2f}",
        "mean_majority_fraction": "{:.2%}",
    }))

    plt.figure(figsize=(8, 4))
    if HAS_SEABORN:
        sns.histplot(data=per_prompt, x="unique_answers", hue="run_id", multiple="stack", discrete=True)
    else:
        per_prompt["unique_answers"].hist(bins=range(1, int(per_prompt["unique_answers"].max()) + 3))
    plt.title("Final Answer Diversity Across Rollouts")
    plt.xlabel("Unique answer keys per prompt")
    plt.ylabel("Prompts")
    plt.tight_layout()
    plt.show()

## Accuracy Variants And Entropy

For best-of-N, entropy is averaged over the rollouts whose answer matches the majority answer. For oracle, it is averaged over correct rollouts when any exist, otherwise all rollouts. For random, it is the selected rollout.

In [ ]:
if variants_df.empty:
    display(variants_df)
else:
    variants_df["correct"] = variants_df["correct"].astype(bool)
    metric_cols = [
        "decoded_token_entropy_mean",
        "answer_option_entropy_mean",
        "hidden_step_entropy_mean",
        "controller_entropy_mean",
    ]
    variant_summary = variants_df.groupby(["run_id", "variant"]).agg(
        prompts=("example_id", "count"),
        accuracy=("correct", "mean"),
        **{col: (col, "mean") for col in metric_cols},
    ).reset_index()
    display(variant_summary.style.format({
        "accuracy": "{:.2%}",
        "decoded_token_entropy_mean": "{:.4f}",
        "answer_option_entropy_mean": "{:.4f}",
        "hidden_step_entropy_mean": "{:.4f}",
        "controller_entropy_mean": "{:.4f}",
    }))

    long_entropy = variants_df.melt(
        id_vars=["run_id", "variant", "correct"],
        value_vars=metric_cols,
        var_name="entropy_metric",
        value_name="entropy",
    ).dropna(subset=["entropy"])
    if not long_entropy.empty:
        plt.figure(figsize=(11, 5))
        if HAS_SEABORN:
            sns.boxplot(data=long_entropy, x="entropy_metric", y="entropy", hue="variant", showfliers=False)
        else:
            long_entropy.boxplot(column="entropy", by=["entropy_metric", "variant"], rot=45)
        plt.title("Selected-Rollout Entropy By Accuracy Variant")
        plt.xlabel("")
        plt.ylabel("Entropy")
        plt.xticks(rotation=25, ha="right")
        plt.tight_layout()
        plt.show()

## Metric-Selected Rollout Accuracy

These selectors pick exactly one rollout per prompt: the rollout with the lowest decoded-token entropy mean, answer-option entropy, hidden-step entropy mean, and an equal-weight normalized combination of those three metrics.


In [ ]:
selector_metric_specs = [
    ("lowest_decoded_entropy", "token_entropy_mean"),
    ("lowest_answer_option_entropy", "answer_option_entropy"),
    ("lowest_hidden_step_entropy", "hidden_step_entropy_mean"),
]
combined_selector = "lowest_equal_normalized_entropy_combo"
combined_metric_cols = [metric_col for _, metric_col in selector_metric_specs]
selector_key_cols = ["run_id", "example_id", "rollout_idx"]


def _choose_lowest_metric(group, selector, metric_col):
    ranked = group.dropna(subset=[metric_col]).sort_values([metric_col, "rollout_idx"], kind="mergesort")
    if ranked.empty:
        return None
    selected = ranked.iloc[0].copy()
    selected["selector"] = selector
    selected["selection_score"] = selected[metric_col]
    selected["selection_metric"] = metric_col
    return selected


def _choose_lowest_normalized_combo(group):
    scored = group.copy()
    normalized_cols = []
    for metric_col in combined_metric_cols:
        if metric_col not in scored:
            continue
        values = pd.to_numeric(scored[metric_col], errors="coerce")
        valid_values = values.dropna()
        if valid_values.empty:
            continue
        spread = valid_values.max() - valid_values.min()
        normalized_col = f"{metric_col}_normalized"
        if pd.isna(spread) or spread == 0:
            scored[normalized_col] = np.where(values.notna(), 0.0, np.nan)
        else:
            scored[normalized_col] = (values - valid_values.min()) / spread
        normalized_cols.append(normalized_col)

    if not normalized_cols:
        return None
    scored["selection_score"] = scored[normalized_cols].mean(axis=1, skipna=True)
    ranked = scored.dropna(subset=["selection_score"]).sort_values(["selection_score", "rollout_idx"], kind="mergesort")
    if ranked.empty:
        return None
    selected = ranked.iloc[0].copy()
    selected["selector"] = combined_selector
    selected["selection_metric"] = " + ".join(combined_metric_cols)
    return selected


if rollouts_df.empty:
    metric_selected_df = pd.DataFrame()
    display(metric_selected_df)
else:
    selector_source_df = rollouts_df.copy()
    if "answer_option_entropy" not in selector_source_df and not entropy_df.empty:
        entropy_metric_cols = [*selector_key_cols, "answer_option_entropy"]
        if all(col in entropy_df for col in entropy_metric_cols):
            selector_source_df = selector_source_df.merge(
                entropy_df[entropy_metric_cols].drop_duplicates(selector_key_cols),
                on=selector_key_cols,
                how="left",
            )

    if "token_entropy_mean" not in selector_source_df and "decoded_token_entropy_mean" in selector_source_df:
        selector_source_df["token_entropy_mean"] = selector_source_df["decoded_token_entropy_mean"]

    selection_rows = []
    for metric_col in combined_metric_cols:
        if metric_col not in selector_source_df:
            selector_source_df[metric_col] = np.nan
    if "answer_key" not in selector_source_df:
        selector_source_df["answer_key"] = None

    selector_required_cols = [*selector_key_cols, "correct", "answer_key", *combined_metric_cols]
    selector_source_df = selector_source_df[selector_required_cols].copy()
    selector_source_df["correct"] = selector_source_df["correct"].astype(bool)
    for metric_col in combined_metric_cols:
        selector_source_df[metric_col] = pd.to_numeric(selector_source_df[metric_col], errors="coerce")

    for _, group in selector_source_df.groupby(["run_id", "example_id"], sort=False):
        for selector, metric_col in selector_metric_specs:
            if metric_col not in group:
                continue
            selected = _choose_lowest_metric(group, selector, metric_col)
            if selected is not None:
                selection_rows.append(selected)
        selected = _choose_lowest_normalized_combo(group)
        if selected is not None:
            selection_rows.append(selected)

    metric_selected_df = pd.DataFrame(selection_rows)
    if metric_selected_df.empty:
        display(metric_selected_df)
    else:
        metric_selected_df = metric_selected_df.rename(columns={
            "rollout_idx": "selected_rollout_idx",
            "answer_key": "selected_answer_key",
        })
        selector_summary = metric_selected_df.groupby(["run_id", "selector"]).agg(
            prompts=("example_id", "count"),
            accuracy=("correct", "mean"),
            mean_selection_score=("selection_score", "mean"),
            mean_decoded_entropy=("token_entropy_mean", "mean"),
            mean_answer_option_entropy=("answer_option_entropy", "mean"),
            mean_hidden_step_entropy=("hidden_step_entropy_mean", "mean"),
        ).reset_index()
        display(selector_summary.style.format({
            "accuracy": "{:.2%}",
            "mean_selection_score": "{:.4f}",
            "mean_decoded_entropy": "{:.4f}",
            "mean_answer_option_entropy": "{:.4f}",
            "mean_hidden_step_entropy": "{:.4f}",
        }))

        plt.figure(figsize=(10, max(4, 0.4 * selector_summary["run_id"].nunique())))
        if HAS_SEABORN:
            sns.barplot(data=selector_summary, y="run_id", x="accuracy", hue="selector")
        else:
            for selector, group in selector_summary.groupby("selector"):
                plt.scatter(group["accuracy"], group["run_id"], label=selector)
            plt.legend()
        plt.gca().xaxis.set_major_formatter(PercentFormatter(1.0))
        plt.title("Metric-Selected Rollout Accuracy By Run")
        plt.xlabel("Accuracy")
        plt.ylabel("")
        plt.tight_layout()
        plt.show()




## Entropy And Correctness

These views are useful for checking whether uncertainty increases on incorrect rollouts, and whether hidden-step entropy tracks controller uncertainty.

In [ ]:
if rollouts_df.empty:
    display(rollouts_df)
else:
    entropy_cols = [
        "token_entropy_mean",
        "hidden_step_entropy_mean",
        "controller_entropy_mean",
    ]
    available_entropy_cols = [col for col in entropy_cols if col in rollouts_df]
    correctness_entropy = rollouts_df.groupby(["run_id", "correct"])[available_entropy_cols].mean().reset_index()
    display(correctness_entropy.style.format({col: "{:.4f}" for col in available_entropy_cols}))

    scatter_df = rollouts_df.dropna(subset=["hidden_step_entropy_mean", "controller_entropy_mean"]).copy()
    if not scatter_df.empty:
        plt.figure(figsize=(7, 5))
        if HAS_SEABORN:
            sns.scatterplot(
                data=scatter_df,
                x="controller_entropy_mean",
                y="hidden_step_entropy_mean",
                hue="correct",
                alpha=0.55,
            )
        else:
            for correct, group in scatter_df.groupby("correct"):
                plt.scatter(group["controller_entropy_mean"], group["hidden_step_entropy_mean"], alpha=0.55, label=f"correct={correct}")
            plt.legend()
        plt.title("Controller Entropy vs Hidden-Step Entropy")
        plt.xlabel("Mean controller entropy")
        plt.ylabel("Mean hidden-step next-token entropy")
        plt.tight_layout()
        plt.show()

## Hidden-Step Entropy Trajectories

The rollout script measures hidden-step entropy after each sampled controller action. This is not prefix rollout tracking; it is one next-token entropy measurement on the actual sampled trajectory.

In [ ]:
step_rows = []
if not rollouts_df.empty and "hidden_step_entropy" in rollouts_df:
    for _, row in rollouts_df.iterrows():
        for metric in row.get("hidden_step_entropy") or []:
            step_rows.append({
                "run_id": row["run_id"],
                "example_id": row["example_id"],
                "rollout_idx": row["rollout_idx"],
                "correct": row["correct"],
                "step_idx": metric.get("step_idx"),
                "action": metric.get("action"),
                "entropy": metric.get("entropy"),
                "retained_probability_mass": metric.get("retained_probability_mass"),
            })

steps_df = pd.DataFrame(step_rows)
print(f"Step entropy rows: {len(steps_df):,}")
if not steps_df.empty:
    trajectory = steps_df.groupby(["run_id", "step_idx", "correct"]).agg(
        observations=("entropy", "count"),
        mean_entropy=("entropy", "mean"),
    ).reset_index()
    display(trajectory.head(30).style.format({"mean_entropy": "{:.4f}"}))

    plt.figure(figsize=(9, 5))
    if HAS_SEABORN:
        sns.lineplot(data=trajectory, x="step_idx", y="mean_entropy", hue="correct", style="run_id", marker="o")
    else:
        for key, group in trajectory.groupby(["run_id", "correct"]):
            plt.plot(group["step_idx"], group["mean_entropy"], marker="o", label=str(key))
        plt.legend()
    plt.title("Hidden-Step Entropy Across Sampled Controller Steps")
    plt.xlabel("Controller step")
    plt.ylabel("Mean next-token entropy")
    plt.tight_layout()
    plt.show()

    action_summary = steps_df.groupby(["action", "correct"]).agg(
        observations=("entropy", "count"),
        mean_entropy=("entropy", "mean"),
    ).sort_values("observations", ascending=False).reset_index()
    display(action_summary.style.format({"mean_entropy": "{:.4f}"}))

## Decoded Token Entropy Trajectories

The entropy sidecar stores `decoded_token_entropies` for every generated answer token. This section explodes those lists so decoded entropy can be analyzed by answer-token position and correctness, rather than only through per-rollout summary means.

In [ ]:
decoded_token_rows = []
if not entropy_df.empty and "decoded_token_entropies" in entropy_df:
    for _, row in entropy_df.iterrows():
        for token_idx, entropy in enumerate(row.get("decoded_token_entropies") or []):
            decoded_token_rows.append({
                "run_id": row["run_id"],
                "example_id": row["example_id"],
                "rollout_idx": row.get("rollout_idx"),
                "correct": bool(row.get("correct")),
                "token_idx": token_idx,
                "entropy": entropy,
                "decoded_answer": row.get("decoded_answer"),
            })

decoded_tokens_df = pd.DataFrame(decoded_token_rows)
print(f"Decoded token entropy rows: {len(decoded_tokens_df):,}")
if not decoded_tokens_df.empty:
    decoded_summary = decoded_tokens_df.groupby(["run_id", "token_idx", "correct"]).agg(
        observations=("entropy", "count"),
        mean_entropy=("entropy", "mean"),
        median_entropy=("entropy", "median"),
    ).reset_index()
    display(decoded_summary.head(40).style.format({"mean_entropy": "{:.4f}", "median_entropy": "{:.4f}"}))

    plt.figure(figsize=(9, 5))
    if HAS_SEABORN:
        sns.lineplot(data=decoded_summary, x="token_idx", y="mean_entropy", hue="correct", style="run_id", marker="o")
    else:
        for key, group in decoded_summary.groupby(["run_id", "correct"]):
            plt.plot(group["token_idx"], group["mean_entropy"], marker="o", label=str(key))
        plt.legend()
    plt.title("Decoded Answer Token Entropy By Token Position")
    plt.xlabel("Decoded token index")
    plt.ylabel("Mean token entropy")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 4))
    if HAS_SEABORN:
        sns.boxplot(data=decoded_tokens_df, x="correct", y="entropy", showfliers=False)
    else:
        decoded_tokens_df.boxplot(column="entropy", by="correct")
    plt.title("Decoded Token Entropy By Rollout Correctness")
    plt.xlabel("Correct")
    plt.ylabel("Token entropy")
    plt.tight_layout()
    plt.show()

## Answer Option Entropy

Answer-option entropy is measured over the A/B/C/D option distribution at the first decoded answer-option token. This section uses the sidecar fields directly, including option probabilities when present.

In [ ]:
if entropy_df.empty or "answer_option_entropy" not in entropy_df:
    print("No answer-option entropy rows loaded.")
else:
    option_df = entropy_df.copy()
    option_df["correct"] = option_df["correct"].astype(bool)
    option_df["answer_option_entropy"] = pd.to_numeric(option_df["answer_option_entropy"], errors="coerce")
    coverage = option_df.groupby("run_id")["answer_option_entropy"].apply(lambda values: values.notna().mean()).reset_index(name="coverage")
    print("Answer-option entropy coverage")
    display(coverage.style.format({"coverage": "{:.2%}"}))

    option_summary = option_df.groupby(["run_id", "correct"]).agg(
        rollouts=("example_id", "count"),
        coverage=("answer_option_entropy", lambda values: values.notna().mean()),
        mean_answer_option_entropy=("answer_option_entropy", "mean"),
        median_answer_option_entropy=("answer_option_entropy", "median"),
    ).reset_index()
    display(option_summary.style.format({
        "coverage": "{:.2%}",
        "mean_answer_option_entropy": "{:.4f}",
        "median_answer_option_entropy": "{:.4f}",
    }))

    plotted = option_df.dropna(subset=["answer_option_entropy"])
    if not plotted.empty:
        plt.figure(figsize=(8, 4))
        if HAS_SEABORN:
            sns.boxplot(data=plotted, x="correct", y="answer_option_entropy", showfliers=False)
        else:
            plotted.boxplot(column="answer_option_entropy", by="correct")
        plt.title("Answer Option Entropy By Rollout Correctness")
        plt.xlabel("Correct")
        plt.ylabel("Answer option entropy")
        plt.tight_layout()
        plt.show()

prob_rows = []
if not entropy_df.empty and "answer_option_probabilities" in entropy_df:
    for _, row in entropy_df.iterrows():
        probabilities = row.get("answer_option_probabilities") or {}
        if not isinstance(probabilities, dict):
            continue
        for option, probability in probabilities.items():
            prob_rows.append({
                "run_id": row["run_id"],
                "example_id": row["example_id"],
                "rollout_idx": row.get("rollout_idx"),
                "correct": bool(row.get("correct")),
                "selected_option": row.get("answer_option_selected_option"),
                "option": option,
                "probability": probability,
            })

option_probs_df = pd.DataFrame(prob_rows)
print(f"Answer-option probability rows: {len(option_probs_df):,}")
if not option_probs_df.empty:
    probability_summary = option_probs_df.groupby(["run_id", "correct", "option"]).agg(
        observations=("probability", "count"),
        mean_probability=("probability", "mean"),
    ).reset_index()
    display(probability_summary.style.format({"mean_probability": "{:.4f}"}))

    plt.figure(figsize=(8, 4))
    if HAS_SEABORN:
        sns.barplot(data=probability_summary, x="option", y="mean_probability", hue="correct")
    else:
        for correct, group in probability_summary.groupby("correct"):
            plt.plot(group["option"], group["mean_probability"], marker="o", label=f"correct={correct}")
        plt.legend()
    plt.title("Mean A/B/C/D Probability By Correctness")
    plt.xlabel("Option")
    plt.ylabel("Mean probability")
    plt.tight_layout()
    plt.show()

## Diagnostics

Inspect prompts with high rollout disagreement, oracle-only recoveries, or confident incorrect majority votes.

In [ ]:
if not rollouts_df.empty:
    disagreement = per_prompt.sort_values(["unique_answers", "majority_fraction"], ascending=[False, True])
    display_cols = ["run_id", "example_id", "rollout_count", "unique_answers", "majority_fraction", "any_correct", "random_accuracy_estimate", "mean_hidden_entropy"]
    print("Highest answer disagreement")
    display(disagreement[display_cols].head(20).style.format({
        "majority_fraction": "{:.2%}",
        "random_accuracy_estimate": "{:.2%}",
        "mean_hidden_entropy": "{:.4f}",
    }))

if not variants_df.empty:
    pivot = variants_df.pivot_table(index=["run_id", "example_id"], columns="variant", values="correct", aggfunc="first")
    pivot = pivot.reset_index()
    oracle_only = pivot[(pivot.get("oracle") == True) & (pivot.get("best_of_n") == False)]
    print(f"Oracle-only recoveries: {len(oracle_only):,}")
    display(oracle_only.head(30))

    confident_wrong = variants_df[
        (variants_df["variant"] == "best_of_n")
        & (~variants_df["correct"].astype(bool))
    ].sort_values("hidden_step_entropy_mean", ascending=True)
    print("Low-hidden-entropy incorrect best-of-N prompts")
    display(confident_wrong[["run_id", "example_id", "selected_answer_key", "selected_rollout_indices", "hidden_step_entropy_mean", "decoded_token_entropy_mean", "answer_option_entropy_mean"]].head(20))